# TMEW-1: Temporal Multimodal Episodic World — Colab Training

This notebook clones the repo, installs dependencies, verifies CUDA, and runs the full curriculum.

**Before running:** Go to *Runtime → Change runtime type → GPU* (T4 or better).

## 1. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/ai-wes/hmwmc.git"
REPO_DIR = "hmwmc"

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR}/ already exists — pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q -r requirements.txt

## 2. Verify CUDA

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device  : {torch.cuda.get_device_name(0)}")
    print(f"Memory  : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → GPU."
    )

## 4. Full curriculum training

In [ ]:
!python tmew1_run.py --epochs 4 --batch-size 8 --train-episodes 2048

## 5. Phase-1 Branch Experiments

Run experimental branches from the research program. Each branch changes one axis
(harder worlds / harder questions / architecture) and auto-evaluates against the
frozen baseline using promotion rubrics.

**Workflow:** Baseline first → then Group 1 branches in parallel.

In [ ]:
# ── Step 1: Run frozen baseline ──────────────────────────────────────
# This produces experiments/baseline/val.json — the reference for all rubrics.
!python tmew1_branch_runner.py --branch baseline --out-dir experiments/baseline

In [ ]:
# ── Step 2: Group 1 branches — all 4 in parallel ────────────────────
import subprocess, sys, time

BASELINE = "experiments/baseline/val.json"

BRANCHES = {
    "A1": "Entity scaling (6 → 8)",
    "A2": "Horizon scaling (T=64 → 96)",
    "C1": "HPM Level-2: 4-slot competitive",
    "C2": "HPM read-mode ablation",
}

procs = {}
for branch in BRANCHES:
    cmd = [
        sys.executable, "tmew1_branch_runner.py",
        "--branch", branch,
        "--baseline-record", BASELINE,
        "--out-dir", f"experiments/{branch}",
    ]
    log = open(f"experiments/{branch}.log", "w")
    procs[branch] = (subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT), log)
    print(f"  launched {branch} — {BRANCHES[branch]}")

print(f"\n{len(procs)} branches running in parallel. Waiting...")

for branch, (proc, log) in procs.items():
    rc = proc.wait()
    log.close()
    status = {0: "promote", 1: "kill", 2: "undecided", 4: "crash"}.get(rc, f"code={rc}")
    print(f"  {branch} finished → {status}")

print("\nAll done. Run the verdict cell below for details.")

In [ ]:
# ── Step 3: Inspect verdicts ─────────────────────────────────────────
import json, glob, os

print(f"{'Branch':<10} {'Verdict':<12} {'Target Metric':<30} {'Value':>8}")
print("─" * 65)
for d in sorted(glob.glob("experiments/*/")):
    branch = os.path.basename(d.rstrip("/\\"))
    verdict_files = glob.glob(os.path.join(d, "verdict_*.jsonl"))
    if not verdict_files:
        print(f"{branch:<10} {'no verdict':<12}")
        continue
    with open(verdict_files[-1]) as f:
        lines = [json.loads(l) for l in f if l.strip()]
        rec = lines[-1] if lines else {}
    v = rec.get("verdict", "?")
    target = rec.get("config", {}).get("rubric", {}).get("target_metric", "—") if rec.get("config") else "—"
    val = rec.get("val_record", {}).get(target, float("nan")) if target != "—" else float("nan")
    print(f"{branch:<10} {v:<12} {target:<30} {val:>8.4f}")

## 6. (Optional) Custom branch with CLI overrides

Tweak any branch parameters without touching config files.

In [ ]:
# Example: A2 with longer horizon and more epochs
!python tmew1_branch_runner.py \
    --branch A2 \
    --tier3-episode-length 128 \
    --epochs 6 \
    --baseline-record experiments/baseline/val.json \
    --out-dir experiments/A2_T128